# SME Capital Matching — Funding Readiness Model & Segmentation

**Notebook 3 of 3.** Like the others, every code cell has a plain-language explanation above it.

**What this notebook does.** Notebook 2 gave us clean data. This notebook *uses* it: we give every SME a **Funding Readiness Score from 0 to 100**, then sort them into three tiers — **High**, **Mid** and **Low** readiness — so the funding team knows who to progress now, who to develop, and who to revisit later.

**How the score is built — four weighted pillars.** Each pillar is scored on a 0-to-1 scale inside the model, multiplied by its weight, and the four are added up to make the 0–100 score:

| Pillar | Weight | The question it answers |
|---|---|---|
| 1. Revenue & Growth | **30 pts** | Is there a real, growing business here? |
| 2. Jobs & Scale | **25 pts** | Is there operating traction and job-creation impact? |
| 3. Funding-Ask Viability | **25 pts** | Is the amount asked for sensible, and is the form complete enough to assess? |
| 4. Transformation (B-BBEE) | **20 pts** | BEE level and inclusive ownership |

**A fairness rule that runs through everything:** a *missing* answer is treated as **neutral** (a middle score), never punished down to zero. We don't penalise a business for a blank the form failed to capture. (The only exception is where a genuinely absent figure — like no recorded revenue at all — sensibly means "pre-revenue.")

**Where this notebook sits in the workflow.** Like Notebook 2, this notebook reads from and writes to the single **`Datasets` folder**. It is the last link in the chain:

| Step | File | Folder |
|---|---|---|
| *Notebook 2 produced this* | `Capital_Matching_Cleaned_Data.xlsx` | `Datasets` |
| **Input** — we open that same file | `Capital_Matching_Cleaned_Data.xlsx` | `Datasets` |
| ⬇ *this notebook scores and tiers every business* | | |
| **Output** — the finished segmentation | `Funding_Readiness_Segmentation.xlsx` | `Datasets` |

So: **we read Notebook 2's Excel file out of the `Datasets` folder, and we save our finished segmentation back into the same folder.** If you have just run Notebook 2, this notebook will pick its output up automatically — there is nothing to move or rename by hand.

This design follows the reference script `build_funding_readiness_model.py` exactly.

## Step 1 — Find the `Datasets` folder, load the clean data, and set the pillar weights

Three things happen here.

**First, we locate the `Datasets` folder** — exactly the same way Notebook 2 did. The notebook looks in the folder it is running from, then the folders above it, until it finds one called `Datasets`. That is the project's single filing cabinet, and it is where both our input and our output live.

**Second, we open Notebook 2's output** — `Capital_Matching_Cleaned_Data.xlsx` — from that folder. This is the direct handover between the two notebooks. If the file isn't there, the notebook stops and tells you to run Notebook 2 first, rather than failing with a confusing error.

**Third, we declare the four pillar weights up front, in one place.** Keeping the weights here (rather than buried in the code) means the model can be re-tuned by changing four numbers — nothing else.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# same as Notebook 2: start where we are and walk upwards until we find the Datasets folder
def find_datasets_folder():
    """Look in this folder, then the one above it, and so on, until we find 'Datasets'."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "Datasets").is_dir():
            return folder / "Datasets"
    raise FileNotFoundError(
        "Could not find a folder named 'Datasets'. Please open this notebook from "
        "inside the project folder (the one that contains 'Datasets')."
    )


DATASETS = find_datasets_folder()                                     # read from here, save to here
CLEAN_FILE = DATASETS / "Capital_Matching_Cleaned_Data.xlsx"          # what Notebook 2 left for us
SEGMENTATION_FILE = DATASETS / "Funding_Readiness_Segmentation.xlsx"  # what we will save at the end

# if Notebook 2 hasn't been run, say so in plain English instead of throwing a cryptic error
if not CLEAN_FILE.exists():
    raise FileNotFoundError(
        f"'{CLEAN_FILE.name}' is not in the Datasets folder yet.\n"
        "Please run Notebook 2 (Data_Cleaning) first - it creates this file."
    )

print(f"Datasets folder : {DATASETS}")
print(f"Reading from    : {CLEAN_FILE.name}   (Notebook 2's output)")
print(f"Will save to    : {SEGMENTATION_FILE.name}")

# load the clean data and pin down the four pillar weights (they add up to 100)
df = pd.read_excel(CLEAN_FILE)
print(f"\nLoaded {len(df):,} clean businesses")

WEIGHTS = {
    "revenue_growth": 30,   # Pillar 1
    "jobs_scale":     25,   # Pillar 2
    "ask_viability":  25,   # Pillar 3
    "transformation": 20,   # Pillar 4
}
assert sum(WEIGHTS.values()) == 100, "weights must total 100"

def clip01(s):
    """Keep a score safely inside the 0..1 range."""
    return s.clip(lower=0, upper=1)

Datasets folder : C:\Users\IC Clearwater\OneDrive\Documents\GitHub\SME_Capital_Funding_Optimization\Datasets
Reading from    : Capital_Matching_Cleaned_Data.xlsx   (Notebook 2's output)
Will save to    : Funding_Readiness_Segmentation.xlsx



Loaded 1,116 clean businesses


## Step 2 — Pillar 1: Revenue & Growth (30 points)

This pillar rewards a business for **(a)** already earning revenue and **(b)** growing. It has two parts:

- **Size (60%)** — where the latest revenue band sits on the 0–6 scale (using 2024, falling back to 2023). Bigger, established revenue scores higher. A business with no revenue band recorded is treated as **pre-revenue** (size 0) rather than dropped.
- **Growth (40%)** — how many bands the business moved between 2023 and 2024. Moving **up** is a strong positive; staying **flat** is neutral (0.5); moving **down** is negative. Unknown growth is treated as neutral.

In [2]:
# Pillar 1 - is there a real, growing business?
def score_revenue_growth(df):
    # SIZE: latest known revenue band, scaled so band 6 = 1.0 (no band recorded = pre-revenue = 0)
    latest_scale = df["revenue_scale_2024"].fillna(df["revenue_scale_2023"])
    size = (latest_scale / 6.0).fillna(0.0)

    # GROWTH: change in band 2023 -> 2024, mapped so flat = 0.5, big jumps capped at +/-2 bands
    delta = df["revenue_scale_2024"] - df["revenue_scale_2023"]
    growth = ((delta.clip(-2, 2)) + 2) / 4.0
    growth = growth.fillna(0.5)            # unknown growth = neutral

    pillar = clip01(0.60 * size + 0.40 * growth)   # blend: 60% size, 40% growth
    df["p1_revenue_size"] = size
    df["p1_revenue_growth"] = growth
    df["pillar_revenue_growth"] = pillar
    return pillar

score_revenue_growth(df)
print("Pillar 1 done. Average (0-1):", round(df["pillar_revenue_growth"].mean(), 3))

Pillar 1 done. Average (0-1): 0.319


## Step 3 — Pillar 2: Jobs & Scale (25 points)

This pillar rewards **existing employment** (proof the business really operates) and **credible job creation** from the funding. We use **log scaling** because the jump from 1 to 5 employees matters far more than 50 to 54. A business of roughly **50 staff or 50 new jobs** hits full marks on this dimension.

- Existing employees: **55%**
- Anticipated new jobs: **45%**

Blanks are treated as zero here (no evidence of jobs), which is the one place the reference script deliberately does not use a neutral middle — an unstated headcount reasonably reads as 'none recorded'.

In [3]:
# Pillar 2 - operating traction + job-creation impact (log-scaled)
def score_jobs_scale(df):
    emp = df["employees"].fillna(0).clip(lower=0)
    jobs = df["jobs_anticipated"].fillna(0).clip(lower=0)

    # log1p compresses the long tail; ~50 is the "full marks" ceiling
    emp_norm = np.log1p(emp) / np.log1p(50)
    jobs_norm = np.log1p(jobs) / np.log1p(50)

    pillar = clip01(0.55 * emp_norm + 0.45 * jobs_norm)   # 55% existing, 45% anticipated
    df["p2_employees_norm"] = clip01(emp_norm)
    df["p2_jobs_norm"] = clip01(jobs_norm)
    df["pillar_jobs_scale"] = pillar
    return pillar

score_jobs_scale(df)
print("Pillar 2 done. Average (0-1):", round(df["pillar_jobs_scale"].mean(), 3))

Pillar 2 done. Average (0-1): 0.409


## Step 4 — Pillar 3: Funding-Ask Viability (25 points)

Two ideas combine here:

- **Sensible ask (60%)** — the amount requested should be proportionate to the size of the business. An ask between **0.5× and 5× current revenue** is the "sweet spot" and scores full marks. A tiny business asking for far more than it earns, or an unusually small ask, scores lower. If we have no revenue figure to compare against, the score stays **neutral** rather than penalising the business.
- **Completeness (40%)** — an application missing key fields is simply harder to fund, so how complete the form is becomes a readiness signal in its own right. We measure the share of ten key assessable fields that are filled in.

In [4]:
# Pillar 3 - is the ask sensible, and is the form complete enough to assess?
def score_ask_viability(df):
    # reference revenue: parsed 2025 revenue, else the 2024 band midpoint, else 2023 midpoint
    ref_rev = df["revenue_2025_zar"]
    ref_rev = ref_rev.where(ref_rev > 0, df["revenue_mid_2024"])
    ref_rev = ref_rev.where(ref_rev > 0, df["revenue_mid_2023"])

    ask = df["funding_ask_zar"]
    ratio = ask / ref_rev            # how many times current revenue is being requested

    def ratio_score(r):
        if pd.isna(r) or r <= 0:
            return np.nan
        if 0.5 <= r <= 5:
            return 1.0               # the sweet spot
        if r < 0.5:
            return max(0.4, r / 0.5) # small asks still fairly fundable
        return max(0.0, 1 - (r - 5) / 20.0)   # very large asks decay towards 0

    sweet = ratio.map(ratio_score).fillna(0.5)   # no revenue reference -> stay neutral

    # completeness across ten key assessable fields
    key_fields = ["company_reg_no", "industry", "province", "funding_ask_zar", "funding_type",
                  "funding_purpose", "revenue_band_2024", "employees", "company_overview", "bee_level"]
    completeness = df[key_fields].notna().mean(axis=1)

    pillar = clip01(0.60 * sweet + 0.40 * completeness)   # 60% sensible ask, 40% completeness
    df["ask_to_revenue_ratio"] = ratio
    df["p3_ask_sweetspot"] = sweet
    df["p3_completeness"] = completeness
    df["pillar_ask_viability"] = pillar
    return pillar

score_ask_viability(df)
print("Pillar 3 done. Average (0-1):", round(df["pillar_ask_viability"].mean(), 3))

Pillar 3 done. Average (0-1): 0.841


## Step 5 — Pillar 4: Transformation / B-BBEE (20 points)

South African funders weigh transformation heavily, so it earns its own pillar. It combines:

- **BEE level (50%)** — Level 1 is best, Level 8 worst. A missing level is treated as neutral.
- **Inclusive ownership (50%)** — the average of the four ownership percentages (Black, women, youth, disability).

**Why it's weighted lowest (20 pts):** this applicant pool is already highly transformed — most are BEE Level 1 and heavily Black-owned — so this pillar barely separates one applicant from another. It's kept because it's a genuine funder criterion, but it does little of the sorting work in practice.

In [5]:
# Pillar 4 - BEE level + inclusive ownership (kept, but low variance in this pool)
def score_transformation(df):
    # BEE level 1..8 -> 1.0..0.0 (Level 1 best). Missing -> neutral 0.5
    bee = df["bee_level"]
    bee_score = ((8 - bee) / 7.0).fillna(0.5).clip(0, 1)

    own_cols = ["black_ownership_pct", "women_ownership_pct", "youth_ownership_pct", "disability_ownership_pct"]
    ownership = (df[own_cols].mean(axis=1) / 100.0).fillna(0.5).clip(0, 1)

    pillar = clip01(0.50 * bee_score + 0.50 * ownership)   # 50/50 blend
    df["p4_bee_score"] = bee_score
    df["p4_ownership"] = ownership
    df["pillar_transformation"] = pillar
    return pillar

score_transformation(df)
print("Pillar 4 done. Average (0-1):", round(df["pillar_transformation"].mean(), 3))

Pillar 4 done. Average (0-1): 0.703


## Step 6 — Add up the four pillars into the 0–100 score

Each pillar (a 0-to-1 number) is multiplied by its weight and the four are summed into the final **Funding Readiness Score**. We also record each pillar's **point contribution** separately — that's what lets us later say *"this business scored well on revenue but lost points on the size of its ask."*

In [6]:
# combine the four pillars into one 0-100 score, and keep each pillar's point contribution
p1 = df["pillar_revenue_growth"]; p2 = df["pillar_jobs_scale"]
p3 = df["pillar_ask_viability"];  p4 = df["pillar_transformation"]

df["funding_readiness_score"] = (
    p1 * WEIGHTS["revenue_growth"] + p2 * WEIGHTS["jobs_scale"]
    + p3 * WEIGHTS["ask_viability"] + p4 * WEIGHTS["transformation"]
).round(1)

df["pts_revenue_growth"] = (p1 * WEIGHTS["revenue_growth"]).round(1)
df["pts_jobs_scale"]     = (p2 * WEIGHTS["jobs_scale"]).round(1)
df["pts_ask_viability"]  = (p3 * WEIGHTS["ask_viability"]).round(1)
df["pts_transformation"] = (p4 * WEIGHTS["transformation"]).round(1)

print("Score summary (0-100):")
print(df["funding_readiness_score"].describe().round(1).to_string())

Score summary (0-100):
count    1116.0
mean       54.9
std        11.8
min        16.5
25%        48.2
50%        54.2
75%        61.7
max        92.0


## Step 7 — Place every business in a tier, and rank them

We turn the score into three plain-language tiers, using fixed thresholds so the tiers keep the same meaning even if new applications are added later:

- **High readiness — score ≥ 62** — the strongest candidates, ready to progress.
- **Mid readiness — 48 to 61** — promising, but need development or more information.
- **Low readiness — below 48** — early-stage or incomplete; not yet fundable.

We also give every business a **rank** (1 = most ready) so the list can be worked from the top down.

In [7]:
# sort every business into High / Mid / Low, and rank them 1 = most ready
def tier(score):
    if score >= 62:
        return "High"
    if score >= 48:
        return "Mid"
    return "Low"

df["readiness_tier"] = pd.Categorical(
    df["funding_readiness_score"].map(tier), categories=["High", "Mid", "Low"], ordered=True
)
df["readiness_rank"] = df["funding_readiness_score"].rank(ascending=False, method="min").astype(int)

# now put the best at the top. plenty of businesses land on exactly the same score, so we settle
# those ties alphabetically by company name. without a tie-breaker the row order would come out
# differently every time the notebook is run - and the same data should always give the same file.
df = df.sort_values(
    ["funding_readiness_score", "company_name"], ascending=[False, True]
).reset_index(drop=True)

tier_counts = df["readiness_tier"].value_counts().reindex(["High", "Mid", "Low"])
tier_table = pd.DataFrame({
    "Tier": tier_counts.index,
    "Businesses": tier_counts.values,
    "Share": [f"{v/len(df)*100:.1f}%" for v in tier_counts.values],
})
display(tier_table)

,Tier,Businesses,Share
0,High,273,24.5%
1,Mid,568,50.9%
2,Low,275,24.6%


## Step 8 — A quick look at the top of the list

Before saving, a sanity check: who came out on top? These are the five most funding-ready businesses in the whole applicant pool. It is worth eyeballing them — if the names at the top look like plausible, substantial businesses, the model is behaving sensibly.

In [8]:
# who's at the top of the pile?
print("Top 5 most funding-ready businesses:")
display(df[["readiness_rank", "company_name", "industry", "province",
            "funding_readiness_score", "readiness_tier"]].head(5))

Top 5 most funding-ready businesses:


,readiness_rank,company_name,industry,province,funding_readiness_score,readiness_tier
0,1,Vuka Transport CC,NPO & NPC,Gauteng,92.0,High
1,2,Maluleke Agri Group,Mining,Limpopo,90.1,High
2,3,Ubuntu Distributors CC,Education,Mpumalanga,90.0,High
3,4,Rakoma Engineering CC,Agriculture - Primary,Limpopo,89.6,High
4,5,Clear Water Foods (Pty) Ltd,Information & Communication Technology,Gauteng,89.4,High


## Step 9 — Save the funding readiness segmentation into the `Datasets` folder

Finally we save the segmentation as a **single, focused Excel table** — `Funding_Readiness_Segmentation.xlsx` — back into the same `Datasets` folder we read the clean data from.

This is deliberately *not* the full clean dataset. It keeps **only the fields that matter for funding readiness**: each business's identity, the inputs the model actually uses (revenue, jobs, ask, transformation), each pillar's point contribution, and the final score, tier and rank. One sheet, one table, colour-coded by tier. All the explanation and the summarised numbers live separately in the methodology document, not here.

**This is the end of the workflow.** The `Datasets` folder now holds all three files in the chain: the raw applications we started from, the cleaned data in the middle, and this finished segmentation — the file the dashboard and the funding team work from.

In [9]:
# save a focused funding-readiness table - only the fields relevant to readiness
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def clean_cell(v):
    """drop the few invisible characters Excel refuses (kept simple; emoji survive)"""
    if not isinstance(v, str):
        return v
    return "".join(ch for ch in v if ch in "\t\n\r" or (
        ord(ch) >= 0x20 and ord(ch) != 0x7f and not (0x80 <= ord(ch) <= 0x9f)
        and not (0xFDD0 <= ord(ch) <= 0xFDEF) and (ord(ch) & 0xFFFF) not in (0xFFFE, 0xFFFF)))

# the funding-readiness-relevant columns, in a sensible reading order
seg_cols = [
    "readiness_rank", "readiness_tier", "funding_readiness_score", "company_name",
    "industry", "province", "city_town", "funding_type", "funding_ask_zar",
    "revenue_band_2023", "revenue_band_2024", "revenue_2025_zar",
    "employees", "jobs_anticipated", "bee_level",
    "black_ownership_pct", "women_ownership_pct", "youth_ownership_pct", "disability_ownership_pct",
    "ask_to_revenue_ratio",
    "pts_revenue_growth", "pts_jobs_scale", "pts_ask_viability", "pts_transformation",
]
seg_cols = [c for c in seg_cols if c in df.columns]
seg = df[seg_cols].copy()

NAVY="1F3864"; BLUE="2E5496"; FONT="Arial"
GREEN="C6EFCE"; YELLOW="FFEB9C"; RED="FFC7CE"; GREEN_T="006100"; YELLOW_T="9C6500"; RED_T="9C0006"
thin=Side(style="thin",color="D9D9D9"); BORDER=Border(left=thin,right=thin,top=thin,bottom=thin)

wb=Workbook(); ws=wb.active; ws.title="Funding Readiness"; ws.sheet_view.showGridLines=False
for j,col in enumerate(seg.columns,1):
    cell=ws.cell(row=1,column=j,value=str(col))
    cell.font=Font(name=FONT,bold=True,color="FFFFFF",size=10)
    cell.fill=PatternFill("solid",fgColor=BLUE)
    cell.alignment=Alignment(horizontal="center",vertical="center",wrap_text=True); cell.border=BORDER

for i,(_,r) in enumerate(seg.iterrows(),2):
    for j,col in enumerate(seg.columns,1):
        v=r[col]
        if pd.isna(v): v=None
        elif isinstance(v,(np.integer,)): v=int(v)
        elif isinstance(v,(np.floating,)): v=float(v)
        cell=ws.cell(row=i,column=j,value=clean_cell(v))
        cell.font=Font(name=FONT,size=9); cell.border=BORDER; cell.alignment=Alignment(vertical="center")
        if col=="funding_ask_zar" or col=="revenue_2025_zar": cell.number_format='#,##0;(#,##0);-'
        if col=="funding_readiness_score": cell.number_format='0.0'
        if col=="ask_to_revenue_ratio": cell.number_format='0.0"x"'
    # colour the tier cell green / amber / red
    tcell=ws.cell(row=i,column=2)
    if tcell.value=="High": tcell.fill=PatternFill("solid",fgColor=GREEN); tcell.font=Font(name=FONT,size=9,bold=True,color=GREEN_T)
    elif tcell.value=="Mid": tcell.fill=PatternFill("solid",fgColor=YELLOW); tcell.font=Font(name=FONT,size=9,bold=True,color=YELLOW_T)
    elif tcell.value=="Low": tcell.fill=PatternFill("solid",fgColor=RED); tcell.font=Font(name=FONT,size=9,bold=True,color=RED_T)
ws.freeze_panes="A2"
for c in ws.columns:
    letter=get_column_letter(c[0].column)
    length=max((len(str(cell.value)) if cell.value is not None else 0) for cell in c)
    ws.column_dimensions[letter].width=min(max(length+2,10),34)

# SEGMENTATION_FILE was set right at the top - it points into the Datasets folder
wb.save(SEGMENTATION_FILE)
print(f"Saved the funding readiness table -> {SEGMENTATION_FILE}")
print(f"({seg.shape[0]:,} rows x {seg.shape[1]} columns)")
print("\nWorkflow complete. The Datasets folder now holds:")
print("  1. capital_matching_applications.csv    (raw input)")
print("  2. Capital_Matching_Cleaned_Data.xlsx   (from Notebook 2)")
print("  3. Funding_Readiness_Segmentation.xlsx  (from this notebook)")

Saved the funding readiness table -> C:\Users\IC Clearwater\OneDrive\Documents\GitHub\SME_Capital_Funding_Optimization\Datasets\Funding_Readiness_Segmentation.xlsx
(1,116 rows x 24 columns)

Workflow complete. The Datasets folder now holds:
  1. capital_matching_applications.csv    (raw input)
  2. Capital_Matching_Cleaned_Data.xlsx   (from Notebook 2)
  3. Funding_Readiness_Segmentation.xlsx  (from this notebook)
